In [1]:
# ============================================================
# 02.08 — RETAIL SALES ANALYTICS
# RETURNS TRANSACTION GENERATOR
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import math
import shutil

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "synthetic"

PRODUCT_FILE = DATA_DIR / "products_master.csv"
SALES_FILES = sorted(DATA_DIR.glob("sales_*.csv"))

RETURN_TOTAL = 118_400
RANDOM_SEED = 2026

np.random.seed(RANDOM_SEED)

print("Project folder:", PROJECT_ROOT)
print("Synthetic data folder:", DATA_DIR)
print("Sales files found:", len(SALES_FILES))

if len(SALES_FILES) == 0:
    raise FileNotFoundError(
        f"No sales_YYYY-MM.csv files found in {DATA_DIR}"
    )

# ------------------------------------------------------------
# 2. LOAD PRODUCT MASTER
#    Used only to identify product category for return reasons
# ------------------------------------------------------------

products = pd.read_csv(PRODUCT_FILE)

required_product_cols = {
    "product_id",
    "category"
}

missing_product_cols = required_product_cols - set(products.columns)

if missing_product_cols:
    raise ValueError(
        f"Missing product columns: {missing_product_cols}"
    )

product_category = (
    products[["product_id", "category"]]
    .drop_duplicates("product_id")
    .set_index("product_id")["category"]
    .to_dict()
)

print("Products loaded:", len(product_category))

# ------------------------------------------------------------
# 3. CHECK SALES COLUMNS
# ------------------------------------------------------------

required_sales_cols = {
    "sales_line_id",
    "order_date",
    "product_id",
    "store_id",
    "quantity",
    "unit_selling_price",
    "discount_pct",
    "payment_method",
    "order_status"
}

first_sales = pd.read_csv(SALES_FILES[0], nrows=5)

missing_sales_cols = required_sales_cols - set(first_sales.columns)

if missing_sales_cols:
    raise ValueError(
        f"Missing required sales columns: {missing_sales_cols}"
    )

print("Sales schema check: PASSED")

# ------------------------------------------------------------
# 4. FIRST PASS
#    Count eligible completed sales lines
#
#    We keep return dates inside the baseline period.
#    Therefore, sales after 2026-08-02 are not selected,
#    because a valid return may occur up to 30 days later.
# ------------------------------------------------------------

BASELINE_RETURN_END = pd.Timestamp("2026-08-31")
LATEST_ELIGIBLE_SALE_DATE = BASELINE_RETURN_END - pd.Timedelta(days=30)

eligible_count = 0

print("\nScanning sales files for eligible completed sales...")

for sales_file in SALES_FILES:

    df = pd.read_csv(
        sales_file,
        usecols=list(required_sales_cols)
    )

    df["order_date"] = pd.to_datetime(
        df["order_date"],
        errors="coerce"
    )

    eligible = df[
        (df["order_status"].astype(str).str.strip().str.lower() == "completed")
        & (df["order_date"].notna())
        & (df["order_date"] <= LATEST_ELIGIBLE_SALE_DATE)
        & (pd.to_numeric(df["quantity"], errors="coerce") > 0)
        & (pd.to_numeric(df["unit_selling_price"], errors="coerce") > 0)
    ]

    eligible_count += len(eligible)

    print(
        f"{sales_file.name}: "
        f"{len(eligible):,} eligible lines"
    )

print("\nTotal eligible sales lines:", f"{eligible_count:,}")

if eligible_count < RETURN_TOTAL:
    raise ValueError(
        f"Not enough eligible sales lines to create "
        f"{RETURN_TOTAL:,} return events."
    )

# ------------------------------------------------------------
# 5. RESERVOIR SAMPLE
#
#    Select EXACTLY 118,400 eligible sales lines without
#    loading all 2+ million sales rows into memory.
# ------------------------------------------------------------

print("\nSelecting exactly 118,400 sales lines...")

reservoir = []
seen = 0

rng = np.random.default_rng(RANDOM_SEED)

for sales_file in SALES_FILES:

    df = pd.read_csv(
        sales_file,
        usecols=list(required_sales_cols)
    )

    df["order_date"] = pd.to_datetime(
        df["order_date"],
        errors="coerce"
    )

    df["quantity"] = pd.to_numeric(
        df["quantity"],
        errors="coerce"
    )

    df["unit_selling_price"] = pd.to_numeric(
        df["unit_selling_price"],
        errors="coerce"
    )

    df["discount_pct"] = pd.to_numeric(
        df["discount_pct"],
        errors="coerce"
    ).fillna(0)

    eligible = df[
        (df["order_status"].astype(str).str.strip().str.lower() == "completed")
        & (df["order_date"].notna())
        & (df["order_date"] <= LATEST_ELIGIBLE_SALE_DATE)
        & (df["quantity"] > 0)
        & (df["unit_selling_price"] > 0)
    ].copy()

    for row in eligible.itertuples(index=False):

        seen += 1

        if len(reservoir) < RETURN_TOTAL:
            reservoir.append(row)

        else:
            random_position = rng.integers(0, seen)

            if random_position < RETURN_TOTAL:
                reservoir[random_position] = row

    print(
        f"Processed {sales_file.name} | "
        f"Eligible scanned: {len(eligible):,}"
    )

returns_source = pd.DataFrame(
    reservoir,
    columns=eligible.columns
)

print(
    "\nSelected sales lines:",
    f"{len(returns_source):,}"
)

if len(returns_source) != RETURN_TOTAL:
    raise ValueError(
        "Return source selection did not produce exactly "
        f"{RETURN_TOTAL:,} records."
    )

# ------------------------------------------------------------
# 6. PREPARE RETURN DATA
# ------------------------------------------------------------

returns_source["category"] = (
    returns_source["product_id"]
    .map(product_category)
    .fillna("Other")
)

# ------------------------------------------------------------
# 7. RETURN DATE
#
#    About 80% within 14 days.
#    Remaining returns occur between 15 and 30 days.
# ------------------------------------------------------------

n_fast = int(round(RETURN_TOTAL * 0.80))
n_late = RETURN_TOTAL - n_fast

days_fast = rng.integers(1, 15, size=n_fast)
days_late = rng.integers(15, 31, size=n_late)

return_days = np.concatenate([
    days_fast,
    days_late
])

rng.shuffle(return_days)

returns_source["return_date"] = (
    returns_source["order_date"]
    + pd.to_timedelta(return_days, unit="D")
)

# ------------------------------------------------------------
# 8. RETURN QUANTITY
#
#    Partial returns allowed.
#    Never greater than original sold quantity.
# ------------------------------------------------------------

sold_quantity = (
    returns_source["quantity"]
    .round()
    .astype(int)
)

return_quantity = np.array([
    rng.integers(1, q + 1)
    for q in sold_quantity
])

returns_source["quantity_returned"] = return_quantity

# ------------------------------------------------------------
# 9. RETURN REASONS BY CATEGORY
# ------------------------------------------------------------

reason_map = {
    "Fashion": [
        "Size/Fit Issue",
        "Customer Changed Mind",
        "Quality Issue"
    ],
    "Sports & Fitness": [
        "Size/Fit Issue",
        "Quality Issue",
        "Customer Changed Mind"
    ],
    "Electronics": [
        "Damaged",
        "Quality Issue",
        "Wrong Product"
    ],
    "Home & Kitchen": [
        "Damaged",
        "Quality Issue",
        "Wrong Product"
    ],
    "Beauty & Personal Care": [
        "Quality Issue",
        "Customer Changed Mind",
        "Other"
    ],
    "Grocery": [
        "Damaged",
        "Wrong Product",
        "Other"
    ]
}

def choose_reason(category):
    reasons = reason_map.get(
        category,
        ["Other"]
    )
    return rng.choice(reasons)

returns_source["return_reason"] = (
    returns_source["category"]
    .apply(choose_reason)
)

# ------------------------------------------------------------
# 10. RETURN STATUS MIX
#
# Refunded  ~72%
# Approved  ~10%
# Rejected  ~12%
# Requested  ~6%
#
# Exact event counts are used.
# ------------------------------------------------------------

status_counts = {
    "Refunded": 85_248,
    "Approved": 11_840,
    "Rejected": 14_208,
    "Requested": 7_104
}

status_values = []

for status, count in status_counts.items():
    status_values.extend([status] * count)

status_values = np.array(status_values)
rng.shuffle(status_values)

returns_source["return_status"] = status_values

# ------------------------------------------------------------
# 11. REFUND METHOD MIX
#
# Original Payment ~82%
# Store Credit    ~15%
# Cash             ~3%
# ------------------------------------------------------------

refund_method_counts = {
    "Original Payment": 97_088,
    "Store Credit": 17_760,
    "Cash": 3_552
}

refund_methods = []

for method, count in refund_method_counts.items():
    refund_methods.extend([method] * count)

refund_methods = np.array(refund_methods)
rng.shuffle(refund_methods)

returns_source["refund_method"] = refund_methods

# ------------------------------------------------------------
# 12. REFUND AMOUNT
#
# Net paid amount:
# quantity × unit selling price × (1 - discount%)
#
# Refund amount is based on returned quantity.
# ------------------------------------------------------------

discount_factor = (
    1 - (returns_source["discount_pct"] / 100)
)

returns_source["refund_amount"] = (
    returns_source["quantity_returned"]
    * returns_source["unit_selling_price"]
    * discount_factor
).round(2)

# ------------------------------------------------------------
# 13. PROCESSED STORE
#
# Return is processed against the original sale store.
# ------------------------------------------------------------

returns_source["processed_store_id"] = (
    returns_source["store_id"]
)

# ------------------------------------------------------------
# 14. UNIQUE RETURN ID
# ------------------------------------------------------------

returns_source["return_id"] = [
    f"RET{i:06d}"
    for i in range(1, RETURN_TOTAL + 1)
]

# ------------------------------------------------------------
# 15. FINAL RETURN COLUMN ORDER
# ------------------------------------------------------------

returns_final = returns_source[
    [
        "return_id",
        "sales_line_id",
        "return_date",
        "quantity_returned",
        "return_reason",
        "return_status",
        "refund_method",
        "refund_amount",
        "processed_store_id"
    ]
].copy()

# ------------------------------------------------------------
# 16. SAVE MONTHLY RETURN FILES
#
#    File name is based on return_date.
# ------------------------------------------------------------

return_files = []

for month, month_df in returns_final.groupby(
    returns_final["return_date"].dt.to_period("M")
):

    output_file = DATA_DIR / f"returns_{month}.csv"

    month_df = month_df.sort_values(
        ["return_date", "return_id"]
    )

    month_df.to_csv(
        output_file,
        index=False
    )

    return_files.append(output_file)

# ------------------------------------------------------------
# 17. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("02.08 RETURNS GENERATOR VALIDATION")
print("=" * 65)

print(
    "Total return records:",
    f"{len(returns_final):,}"
)

print(
    "Unique return IDs:",
    returns_final["return_id"].nunique()
)

print(
    "Unique sales_line IDs:",
    returns_final["sales_line_id"].nunique()
)

print(
    "Return files created:",
    len(return_files)
)

print(
    "Minimum return date:",
    returns_final["return_date"].min().date()
)

print(
    "Maximum return date:",
    returns_final["return_date"].max().date()
)

# Quantity validation
quantity_check = (
    returns_source["quantity_returned"]
    <= returns_source["quantity"]
).all()

print(
    "Returned quantity <= sold quantity:",
    quantity_check
)

# Return date validation
date_check = (
    returns_final["return_date"]
    > returns_source["order_date"]
).all()

print(
    "Return date after sale date:",
    date_check
)

# Required field validation
missing_check = returns_final.isna().sum().sum()

print(
    "Missing required values:",
    missing_check
)

# Status distribution
print("\nReturn status distribution:")
print(
    returns_final["return_status"]
    .value_counts()
)

# Refund method distribution
print("\nRefund method distribution:")
print(
    returns_final["refund_method"]
    .value_counts()
)

# ------------------------------------------------------------
# 18. FINAL CHECK
# ------------------------------------------------------------

all_checks = [
    len(returns_final) == RETURN_TOTAL,
    returns_final["return_id"].nunique() == RETURN_TOTAL,
    returns_final["sales_line_id"].nunique() == RETURN_TOTAL,
    quantity_check,
    date_check,
    missing_check == 0
]

if all(all_checks):

    print("\n" + "=" * 65)
    print("✅ 02.08 RETURNS GENERATOR COMPLETED SUCCESSFULLY")
    print("=" * 65)

    print(
        f"Exactly {RETURN_TOTAL:,} return events generated."
    )

    print(
        "Return files saved in:",
        DATA_DIR
    )

else:

    print("\n❌ VALIDATION FAILED — DO NOT MARK 02.08 COMPLETE.")

print("\nGenerated return files:")

for file in sorted(return_files):
    print(
        f"  {file.name} | "
        f"{len(pd.read_csv(file)):,.0f} rows"
    )

Project folder: c:\Users\HP\OneDrive\Desktop\retail_sales_analytics
Synthetic data folder: c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic
Sales files found: 32
Products loaded: 2500
Sales schema check: PASSED

Scanning sales files for eligible completed sales...
sales_2024-01.csv: 51,068 eligible lines
sales_2024-02.csv: 48,415 eligible lines
sales_2024-03.csv: 53,818 eligible lines
sales_2024-04.csv: 54,802 eligible lines
sales_2024-05.csv: 56,427 eligible lines
sales_2024-06.csv: 58,070 eligible lines
sales_2024-07.csv: 55,345 eligible lines
sales_2024-08.csv: 58,067 eligible lines
sales_2024-09.csv: 61,868 eligible lines
sales_2024-10.csv: 72,541 eligible lines
sales_2024-11.csv: 78,057 eligible lines
sales_2024-12.csv: 86,058 eligible lines
sales_2025-01.csv: 55,134 eligible lines
sales_2025-02.csv: 52,216 eligible lines
sales_2025-03.csv: 58,051 eligible lines
sales_2025-04.csv: 59,232 eligible lines
sales_2025-05.csv: 60,919 eligible lines
sales_2025-06.csv: 6

In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.cwd().parent / "data" / "synthetic"

return_files = sorted(DATA_DIR.glob("returns_*.csv"))

total_rows = 0
all_return_ids = []
all_sales_line_ids = []

for file in return_files:
    df = pd.read_csv(file)

    total_rows += len(df)
    all_return_ids.extend(df["return_id"].astype(str))
    all_sales_line_ids.extend(df["sales_line_id"].astype(str))

print("=" * 60)
print("02.08 FINAL VERIFICATION")
print("=" * 60)

print("Return files:", len(return_files))
print("Total return records:", f"{total_rows:,}")
print("Unique return IDs:", len(set(all_return_ids)))
print("Unique sales_line IDs:", len(set(all_sales_line_ids)))

print("\nExpected return records: 118,400")

if (
    total_rows == 118400
    and len(set(all_return_ids)) == 118400
    and len(set(all_sales_line_ids)) == 118400
):
    print("\n✅ 02.08 VERIFIED — COMPLETE")
else:
    print("\n❌ 02.08 VERIFICATION FAILED")

02.08 FINAL VERIFICATION
Return files: 32
Total return records: 118,400
Unique return IDs: 118400
Unique sales_line IDs: 118400

Expected return records: 118,400

✅ 02.08 VERIFIED — COMPLETE
